# Small Animal Classifier Sagemaker Serverless Deployment
This notebook deploys the [small-animal-classifier](https://github.com/agentmorris/small-animal-classifier/), trained and provided by Dan Morris, to a Sagemaker serverless endpoint. It is intended to be run in a SageMaker Notebook instance on the conda_pytorch_p10 kernel.

## Setup

In [2]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import time
import json
import base64
from datetime import datetime

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


## Intialize AWS Session

In [3]:
session = boto3.Session()
sm = session.client('sagemaker')
region = session.region_name
account = boto3.client('sts').get_caller_identity().get('Account')

## Get IAM Role

Note: Ensure the IAM role has:

- `AmazonS3FullAccess`
- `AmazonSageMakerFullAccess`

In [4]:
role = sagemaker.get_execution_role()
print(f"Using role: {role}")

Using role: arn:aws:iam::830244800171:role/service-role/AmazonSageMaker-ExecutionRole-20210125T212674


## Create ECR Repository

In [5]:
# Create ECR repository if it doesn't exist
registry_name = "small-animal-classifier-sagemaker-serverless"
ecr = boto3.client('ecr')

try:
    ecr.create_repository(repositoryName=registry_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ECR repository already exists")
    pass

ECR repository already exists


## Build and Upload Container

Builds the the inference container with small-animal-classifier and the sagemaker handler and uploads it to ECR. This step takes several minutes after it prints the 'Login Succeeded' message. Be patient and trust the process.

In [6]:
# flag to avoid timely image builds
should_create = True

if should_create:
    # Get auth token and login to ECR
    !aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account}.dkr.ecr.{region}.amazonaws.com
    
    # Build container
    !docker build -q -t {registry_name} -f Dockerfile .
    
    # Tag and push to ECR
    image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{registry_name}:latest"
    !docker tag {registry_name} {image_uri}
    !docker push {image_uri}
    
    print(f"Container pushed to: {image_uri}")

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
sha256:02ee65cf025e63dd1f9bc6e02066be5fe4effc2e0ca24cf238df8a145b7bb386
The push refers to repository [830244800171.dkr.ecr.us-west-2.amazonaws.com/small-animal-classifier-sagemaker-serverless]

bf18a086: Preparing 
8916478c: Preparing 
df18bf8e: Preparing 
c109eddb: Preparing 
aa020374: Preparing 
90588115: Preparing 
00d1b976: Preparing 
ddf62be3: Preparing 
0a9e4758: Preparing 
latest: digest: sha256:e9a7c3dbdf730abb564d0b39e2b7d6fb27fb9e2f91140ef362256827e6c89e8f size: 2420
Container pushed to: 830244800171.dkr.ecr.us-west-2.amazonaws.com/small-animal-classifier-sagemaker-serverless:latest


## Create Sagemaker Model

In [7]:
model_prefix = "small-animal-classifier"

# Check if model already exists
model_already_created = False
for model_def in sm.list_models()['Models']:
    if model_prefix == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True

# Create model if it doesn't exist
if not model_already_created:
    create_model_response = sm.create_model(
        ModelName=model_prefix,
        ExecutionRoleArn=role,
        PrimaryContainer={
            "Image": image_uri,
            "Environment": {
                "SAGEMAKER_PROGRAM": "serve.py"
            }
        }
    )

print(f"Model ARN: {create_model_response['ModelArn']}")

Model ARN: arn:aws:sagemaker:us-west-2:830244800171:model/small-animal-classifier


## Create Sagemaker Realtime Endpoint Configuration

In [9]:
# Create realtime and batch endpoint configuration
realtime_endpoint_config_name = f"{model_prefix}-realtime-config"

realtime_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=realtime_endpoint_config_name,
    ProductionVariants=[
        {
            "ModelName": model_prefix,
            "VariantName": "AllTraffic",
            "ServerlessConfig": {
                "MemorySizeInMB": 6144,  # 6GB memory
                "MaxConcurrency": 20       # Maximum concurrent invocations
            }
        }
    ]
)
print(f"Realtime endpoint config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")

Realtime endpoint config ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint-config/small-animal-classifier-realtime-config


## Create Realtime Endpoint

In [12]:
# Create realtime endpoint
realtime_endpoint_name = f"{model_prefix}-concurrency-20"
create_realtime_endpoint_response = sm.create_endpoint(
    EndpointName=realtime_endpoint_name,
    EndpointConfigName=realtime_endpoint_config_name
)

print(f"Endpoint ARN: {create_realtime_endpoint_response['EndpointArn']}")

# Wait for endpoint creation
resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
realtime_status = resp['EndpointStatus']
print(f"Status: {realtime_status}")

while realtime_status == 'Creating':
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
    realtime_status = resp['EndpointStatus']
    print(f"Status: {realtime_status}")
    if realtime_status == 'Failed':
        realtime_failure_reason = resp.get('FailureReason', 'No failure reason provided')
        print(f"Realtime endpoint deployment failed: {realtime_failure_reason}")
        break

# Get CloudWatch logs for the endpoint
logs = boto3.client('logs')

print(f"Realtime Arn: {resp['EndpointArn']}")
print(f"Realtime endpoint final status: {realtime_status}")
if realtime_status == 'Failed':
    realtime_log_group = f"/aws/sagemaker/Endpoints/{realtime_endpoint_name}"
    try:
        log_streams = logs.describe_log_streams(
            logGroupName=realtime_log_group,
            orderBy='LastEventTime',
            descending=True,
            limit=1
        )
        if log_streams['logStreams']:
            stream = log_streams['logStreams'][0]
            print(f"\nLog stream: {stream['logStreamName']}")
            realtime_events = logs.get_log_events(
                logGroupName=realtime_log_group,
                logStreamName=stream['logStreamName'],
                startFromHead=True
            )
            for event in realtime_events['events']:
                print(event['message'])
    except Exception as e:
        print(f"Error fetching logs: {str(e)}")

Endpoint ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint/small-animal-classifier-concurrency-20
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Failed
Realtime endpoint deployment failed: Unable to successfully stand up your model within the allotted 180 second timeout. Please ensure that downloading your model artifacts, starting your model container and passing the ping health checks can be completed within 180 seconds.
Realtime Arn: arn:aws:sagemaker:us-west-2:830244800171:endpoint/small-animal-classifier-concurrency-20
Realtime endpoint final status: Failed

Log stream: AllTraffic/375c68615008b97b534f59f139e9190b-0c1121f4bf874e73a75822c3ff3b9146
2026-08-17 03:20:55,204 - __main__ - INFO - Loading model from /opt/ml/model/eva02-20260630-llrd.best.e